In [24]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
!which python


2.6.0+cu124
False
/home/rebal/anaconda3/envs/KE-CW2/bin/python


In [26]:
import transformers
print("Transformers version:", transformers.__version__)


Transformers version: 4.50.3


In [1]:
!pip install transformers rdflib
!pip install pandas langdetect deepl transformers rdflib python-dotenv torch




## Lyrics Theme Extraction Pipeline
# 
 This notebook will:
 1. Load a CSV database of song lyrics.
 2. Use DeepL to detect the language and translate non-English lyrics into English.
 3. Use Nyckel’s sentiment analysis API to extract the theme from the (translated) lyrics.
 4. Save the enriched data (with `hasLanguage` and `hasTheme`) to a new CSV file.


In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
import deepl  # Make sure you have your DeepL API key
from langdetect import detect
import requests
from transformers import pipeline
from rdflib import Graph, URIRef, Literal, Namespace
# Load environment variables from .env
load_dotenv()
DEEPL_AUTH_KEY = os.getenv("DEEPL_AUTH_KEY")


/home/rebal/anaconda3/envs/KE-CW2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
translator = deepl.Translator(DEEPL_AUTH_KEY)

# Function to Translate Lyrics Using DeepL

 DeepL automatically detects the source language and returns both the detected language and the translated text.


In [13]:
def translate_lyrics(text):
    try:
        result = translator.translate_text(text, target_lang="EN-US")
        detected_lang = result.detected_source_lang  # Correct attribute name
        translated_text = result.text
        return detected_lang, translated_text
    except Exception as e:
        print("Translation error:", e)
        return "unknown", text


# Function to Extract Song Theme Using Nyckel’s Sentiment Analysis API

 This function calls the Nyckel API endpoint to extract the theme of the song from its lyrics.
 Replace the URL and parameters below with the actual details if available.


In [5]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="bhadresh-savani/distilbert-base-uncased-emotion",
    framework="pt"    # explicitly specify PyTorch
)

Device set to use cpu


In [6]:


def extract_theme(text):
    try:
        # Get predictions from the model
        results = emotion_classifier(text)
        if results:
            # Pick the label with the highest score
            top = max(results, key=lambda x: x['score'])
            return top['label']
        else:
            return "unknown"
    except Exception as e:
        print("Theme extraction error:", e)
        return "unknown"

# Load the Lyrics Database
 
 Assume your CSV file `lyrics.csv` has at least the following columns:
 - `id`
 - `title`
 - `lyrics`


In [7]:
df = pd.read_csv("sample_lyrics.csv")
print(f"Loaded {len(df)} lyric entries.")


Loaded 10 lyric entries.


 ### Process Each Lyric Entry

 For each entry, we:
 1. Use DeepL to detect language and translate lyrics (if necessary).
 2. Use Nyckel to extract the song theme.
 3. Store the detected language, translated lyrics, and theme.


In [14]:
# Create new columns in the dataframe
df["detected_language"] = None
df["translated_lyrics"] = None
df["theme"] = None

for idx, row in df.iterrows():
    lyrics = row["lyrics"]
    # Detect language and translate
    detected_lang, translated = translate_lyrics(lyrics)
    df.at[idx, "detected_language"] = detected_lang
    
    # For theme analysis, if lyrics are already in English, use original; otherwise, use translated version
    text_for_analysis = lyrics if detected_lang.upper() == "EN" else translated
    df.at[idx, "translated_lyrics"] = translated
    
    # Extract theme using the emotion classifier
    theme = extract_theme(text_for_analysis)
    df.at[idx, "theme"] = theme

# Save the Enriched Data

The enriched data (with detected language and theme) will be saved to a new CSV file.


In [16]:
df.to_csv("lyrics_enriched.csv", index=False)
print("Enriched lyrics saved to 'lyrics_enriched.csv'.")
df.head(10)

Enriched lyrics saved to 'lyrics_enriched.csv'.


,id,title,lyrics,detected_language,translated_lyrics,theme
0,1,Imagine,"Imagine there's no heaven, it's easy if you try.",EN,"Imagine there's no heaven, it's easy if you try.",joy
1,2,Bohemian Rhapsody,Is this the real life? Is this just fantasy? C...,EN,Is this the real life? Is this just fantasy? C...,fear
2,3,Hotel California,"On a dark desert highway, cool wind in my hair.",EN,"On a dark desert highway, cool wind in my hair.",joy
3,4,Hey Jude,"Hey Jude, don't make it bad. Take a sad song a...",EN,"Hey Jude, don't make it bad. Take a sad song a...",sadness
4,5,Like a Rolling Stone,"Once upon a time you dressed so fine, threw th...",EN,"Once upon a time you dressed so fine, threw th...",joy
5,6,Smells Like Teen Spirit,"Load up on guns, bring your friends. It's fun ...",EN,"Load up on guns, bring your friends. It's fun ...",joy
6,7,What's Going On,"Mother, mother, there's too many of you crying...",EN,"Mother, mother, there's too many of you crying...",sadness
7,8,Shake It Off,"I stay out too late, got nothing in my brain. ...",EN,"I stay out too late, got nothing in my brain. ...",anger
8,9,Rolling in the Deep,"There's a fire starting in my heart, reaching ...",EN,"There's a fire starting in my heart, reaching ...",fear
9,10,La Bamba,"Para bailar la Bamba, se necesita una poca de ...",ES,"To dance the Bamba, you need a little grace.",joy


RDF Triples

In [18]:
MO = Namespace("http://example.org/mo#")
EX = Namespace("http://example.org/ex#")

# Create RDF graph
g = Graph()
g.bind("mo", MO)
g.bind("ex", EX)

# Create RDF triples for each song entry
for idx, row in df.iterrows():
    # Create a unique URI for the song (e.g., using the song ID)
    song_uri = URIRef(EX["song_" + str(row["id"])])
    
    # Add detected language triple
    g.add((song_uri, MO.hasLanguage, Literal(row["detected_language"])))
    
    # Add theme triple
    g.add((song_uri, MO.hasTheme, Literal(row["theme"])))
    

rdf_output = g.serialize(format="turtle")
with open("lyrics_kg.ttl", "w", encoding="utf-8") as f:
    f.write(rdf_output)
print("RDF triples saved to 'lyrics_kg.ttl'.")



RDF triples saved to 'lyrics_kg.ttl'.
